In [1]:
import json
import pandas as pd
import re

def clean_ingredients(ingredients):
    """Convert ingredients to a clean list."""
    if isinstance(ingredients, str):
        return [i.strip() for i in ingredients.split('\n') if i.strip() and not i.lower().startswith('for ')]
    elif isinstance(ingredients, list):
        return [i.strip() for i in ingredients if i.strip() and not i.lower().startswith('for ')]
    return []

def clean_instructions(instructions):
    """Ensure instructions are a clean list of steps."""
    if isinstance(instructions, str):
        return [step.strip() for step in instructions.split('.') if step.strip()]
    return [step.strip() for step in instructions if step.strip()]

def normalize_allrecipes(data):
    return {
        "name": data.get("title"),
        "ingredients": clean_ingredients(data.get("ingredients", [])),
        "instructions": clean_instructions(data.get("instructions", [])),
        "region": "International",
        "flavor_profile": None,
        "course": None
    }
def normalize_hebbars(data):
    return {
        "name": data.get("name"),
        "ingredients": clean_ingredients(data.get("ingredients", "")),
        "instructions": clean_instructions(data.get("instructions", [])),
        "region": "Indian",
        "flavor_profile": None,
        "course": None
    }
def normalize_tarla(data):
    return {
        "name": data.get("name"),
        "ingredients": clean_ingredients(data.get("ingredients", [])),
        "instructions": clean_instructions(data.get("instructions", [])),
        "region": data.get("region", "Indian"),
        "flavor_profile": data.get("flavor_profile", None),
        "course": None
    }
def normalize_kaggle_row(row):
    return {
        "name": row['name'],
        "ingredients": clean_ingredients(row['ingredients']),
        "instructions": [],  # Not present in Kaggle
        "region": row.get('region'),
        "flavor_profile": row.get('flavor_profile'),
        "course": row.get('course')
    }

In [2]:
# Example loading and parsing JSONs
with open('allrecipes.json') as f:
    allrecipes = [normalize_allrecipes(item) for item in json.load(f)]

with open('hebbars.json') as f:
    hebbars = [normalize_hebbars(item) for item in json.load(f)]

with open('tarladalal.json') as f:
    tarla = [normalize_tarla(item) for item in json.load(f)]

# For Kaggle CSV
df_kaggle = pd.read_csv("kaggle_recipes.csv")
kaggle_data = [normalize_kaggle_row(row) for idx, row in df_kaggle.iterrows()]

# Combined dataset
final_recipes = allrecipes + hebbars + tarla + kaggle_data


In [3]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.3 MB/s eta 0:00:00


In [7]:
import re
from rapidfuzz import fuzz
import pandas as pd

def extract_variants(title):
    """Split by | and clean each variant."""
    parts = title.split('|')
    return [clean_title(p) for p in parts]

def clean_title(name):
    name = name.lower()
    name = re.sub(r'\b(recipe|how to make|at home)\b', '', name)
    name = re.sub(r'[^\w\s]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

def extract_variants(title):
    """Split by | and clean each variant."""
    if not title:
        return []
    parts = title.split('|')
    return [clean_title(p) for p in parts]

def deduplicate_recipes(recipes, threshold=90):
    seen = []
    unique = []

    for recipe in recipes:
        raw_name = recipe.get('title') or recipe.get('name') or ""
        variants = extract_variants(raw_name)

        match_found = False
        for variant in variants:
            for seen_name in seen:
                score = fuzz.ratio(variant, seen_name)
                if score >= threshold:
                    match_found = True
                    break
            if match_found:
                break

        if not match_found:
            seen.extend(variants)
            unique.append(recipe)

    return unique



In [8]:
# Assuming combined dataset
deduplicated = deduplicate_recipes(final_recipes, threshold=90)

print("Original:", len(final_recipes))
print("Deduplicated:", len(deduplicated))


Original: 2873
Deduplicated: 2572


In [9]:
df = pd.DataFrame(deduplicated)
missing_region_df = df[df['region'].isna() | (df['region'].str.strip() == '')]
print(f"Total missing 'region' entries: {len(missing_region_df)}")
print("Existing region labels:", df['region'].dropna().unique())
# Replace any non-empty string like 'None' with actual NaN if needed
df['region'].replace('None', pd.NA, inplace=True)
# Now confirm if missing entries are truly NaN or None
missing_region_df = df[df['region'].isna()]
print(f"Total missing 'region' entries: {len(missing_region_df)}")

Total missing 'region' entries: 1
Existing region labels: ['International' 'Indian' 'South Indian' 'North Indian' 'Bengal'
 'Gujarati' 'Rajasthani' 'Maharashtrian' 'Punjabi' 'East' 'West' 'North'
 '-1' 'North East' 'South' 'Central']
Total missing 'region' entries: 1


In [10]:
import pandas as pd

# Check if the dataset is a DataFrame
if isinstance(deduplicated, pd.DataFrame):
    print("It's already a DataFrame!")
else:
    # If it's a list or other format, convert it to a DataFrame
    deduplicated_data = pd.DataFrame(deduplicated)

In [11]:
# Save the deduplicated dataset as JSON
deduplicated_data.to_json('deduplicated_recipes.json', orient='records', lines=True)


NameError: name 'deduplicated_data' is not defined

In [4]:
import pandas as pd
import re
# Load line-delimited JSONs (JSONL)
df = pd.read_json("deduplicated_recipes.json", lines=True)

In [13]:
import re

def preprocess_recipe(instructions):
    # Convert list to string if needed
    if isinstance(instructions, list):
        instructions = " ".join(instructions)

    def convert_large_numbers_to_fraction(ingredient):
        fraction_map = {
            12: "1/2",
            14: "1/4",
            34: "3/4",
        }
        if ingredient.isdigit():
            num = int(ingredient)
            if num in fraction_map:
                return fraction_map[num]
        return ingredient

    # Replace numbers with fractions using the map
    instructions = re.sub(r'\d+', lambda x: convert_large_numbers_to_fraction(x.group()), instructions)
    return instructions


In [14]:
df['ingredients'] = df['ingredients'].apply(preprocess_recipe)

In [15]:
df.to_json("cleaned_recipes.json", orient="records", lines=True)

###Additional Steps (if model ain't Mistral 7b)

In [ ]:
!pip install spacy
!pip install nltk
!pip install numpy
!pip install pandas

In [ ]:
import pandas as pd

# Load line-delimited JSONs (JSONL)
df = pd.read_json("deduplicated_recipes.json", lines=True)


In [ ]:
!pip install -U spacy
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 30.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import spacy
import re

# Load medium model for better entity recognition (requires !python -m spacy download en_core_web_md)
nlp = spacy.load("en_core_web_md")  # Medium model has better NER for quantities

def preprocess_recipe(instructions):
    """
    Advanced preprocessing for recipe data that:
    - Preserves quantities, temperatures, and units
    - Lemmatizes action verbs and ingredients
    - Handles both strings and list inputs
    - Converts large quantities (greater than 10) into fraction format (only for integers)
    """
    # Convert list to string if needed
    if isinstance(instructions, list):
        instructions = " ".join(instructions)

    # Function to convert numbers > 10 into fractions (e.g., 12 -> 1/2)
    def convert_large_numbers_to_fraction(ingredient):
        # Check if the ingredient is a valid integer and greater than 10
        if ingredient.isdigit() and int(ingredient) > 10:
            tens_digit = int(ingredient) // 10
            ones_digit = int(ingredient) % 10
            if ones_digit != 0:
                return f"{ones_digit}/10"  # Convert to fraction
        return ingredient  # Return as is if not an integer or not > 10

    # Apply the fraction conversion to numbers greater than 10
    instructions = re.sub(r'\d+', lambda x: convert_large_numbers_to_fraction(x.group()), instructions)

    # Pre-protect special number formats before spaCy processing
    instructions = re.sub(
        r'(\d+\.?\d*)\s*(°[FC]|mins?|hours?|cups?|tbsp?|teaspoons?|grams?|ml|lbs?)',
        lambda m: f"QTY_{m.group(1)}_UNIT_{m.group(2)}",
        str(instructions)
    )

    doc = nlp(instructions.lower())

    processed_tokens = []
    for token in doc:
        # Case 1: Preserve protected quantity-unit pairs
        if token.text.startswith("QTY_") and "_UNIT_" in token.text:
            processed_tokens.append(token.text)
            continue

        # Case 2: Keep detected quantities and measurements
        if token.ent_type_ in ["CARDINAL", "QUANTITY", "TIME"]:
            processed_tokens.append(f"NUM_{token.text}")
            continue

        # Case 3: Preserve cooking units not caught by regex
        if token.text in {'cup', 'tsp', 'tbsp', 'gram', 'ounce', 'minute', 'hour', 'degree'}:
            processed_tokens.append(f"UNIT_{token.text}")
            continue

        # Case 4: Lemmatize important content words
        if (token.pos_ in ["NOUN", "VERB", "ADJ"] and
            not token.is_stop and
            not token.is_punct and
            len(token.text) > 2):
            processed_tokens.append(token.lemma_)

    return " ".join(processed_tokens)

# Sample usage
sample_text = "Mix 12 cups flour at 350°F for 30 minutes with 13 tsp salt"
print(preprocess_recipe(sample_text))
# Expected output: mix QTY_12_UNIT_cups flour at QTY_350_UNIT_°f NUM_30 UNIT_minute with QTY_1/3_UNIT_tsp salt

# Test with the provided data for besan chilla
besan_chilla_text = "To make besan chilla, combine the besan, chilli powder, turmeric powder, asafoetida, salt and approx. ¾ cup of water in a deep bowl and whisk well."
print(preprocess_recipe(besan_chilla_text))
# Expected: to make besan chilla combine besan chilli powder turmeric powder asafoetida salt approx QTY_3/4_UNIT_cup water deep bowl whisk well


mix NUM_2 qty_10_unit_cup flour qty_350_unit_ qty_30_unit_minute NUM_3/10 UNIT_tsp salt
besan chilla combine besan chilli powder turmeric powder salt approx UNIT_cup water deep bowl whisk


In [ ]:
df['cleaned_instruction'] = df['instructions'].apply(preprocess_recipe)
df['name'] = df['name'].apply(preprocess_recipe)
df['cleaned_ingredients'] = df['ingredients'].apply(preprocess_recipe)

In [ ]:
# ✅ STEP 7: Save and Download Cleaned File
df.to_json("cleaned_recipes.json", orient="records", lines=True)

In [ ]:
sample = df.sample(50)
for idx, row in sample.iterrows():
    print(f"Original: {row['instructions']}")
    print(f"Processed: {row['cleaned_instruction']}\n---")

Original: []
Processed: 
---
Original: ['For bhapa doi', 'To make bhapa doi, combine the condensed milk , curds and cornflour in a bowl and whisk well till no lumps remain.', 'Pour the mixture into 12 small microwave safe bowls and microwave 4 bowls at a time on high for 40 seconds and serve the bhapa doi immediately or chilled garnished with cardamom powder.', 'Variation:', 'Orange flavoured bhapa doi- after step 1 add 3 tbsp of orange crush and a few drops of lemon juice, mix well. Proceed as per the recipe and serve immediately or chilled garnished with a little orange crush.']
Processed: bhapa doi bhapa doi combine condense milk curd cornflour bowl whisk lump remain pour mixture NUM_2/10 small microwave safe bowl microwave NUM_4 bowl time high NUM_40 NUM_seconds serve bhapa doi chill garnish cardamom powder variation flavour bhapa doi- step NUM_1 add drop lemon juice mix proceed recipe serve chill garnish little orange crush
---
Original: ['Gather all ingredients.', 'Whisk together